# 📈 Regresión lineal con matrices: la función de consumo

**Propedéutico de Economía — FLACSO**

En esta clase vamos a estimar una **función de consumo** por Mínimos Cuadrados
Ordinarios (MCO) usando **álgebra matricial pura**, con la librería del curso
[`algebra-lineal-sheets`](https://pypi.org/project/algebra-lineal-sheets/).
Después **replicaremos el mismo cálculo "desde cero" con pandas y NumPy**,
para comprobar que no hay magia: solo matrices.

**Plan de la clase:**

1. Preparar el entorno local (`.venv`)
2. Cargar y explorar los datos de consumo
3. Teoría: MCO en forma matricial
4. Estimar con `algebra_lineal` (matriz de diseño + ecuaciones normales)
5. Medir el ajuste ($R^2$) y graficar
6. Replicar todo con pandas + NumPy
7. Exportar resultados a Excel
8. Ejercicios

## 0️⃣ Preparación del entorno (una sola vez)

Vamos a trabajar **en local** (VS Code o Jupyter), con un **entorno virtual**
`.venv`: una instalación de Python aislada para esta clase, que no mezcla
librerías con otros proyectos de tu computador.

**Paso 1.** Crea una carpeta para la clase y guarda dentro este notebook y el
archivo [`datos_consumo.csv`](https://raw.githubusercontent.com/franperezec/algebra-lineal-sheets/main/ejemplos/datos_consumo.csv)
(clic derecho → *"Guardar enlace como..."*). Si clonaste el repositorio del
curso, ya tienes ambos en la carpeta `ejemplos/`.

**Paso 2.** Abre la terminal en esa carpeta (en VS Code: `Ctrl` + `` ` ``) y
crea y activa el entorno virtual:

```bash
python -m venv .venv

# Activarlo:
.venv\Scripts\activate        # Windows
source .venv/bin/activate     # macOS / Linux
```

**Paso 3.** Con el entorno activado (verás `(.venv)` al inicio de la línea de
la terminal), instala las librerías de la clase:

```bash
pip install algebra-lineal-sheets pandas matplotlib ipykernel
```

| Librería | Para qué la usamos |
|---|---|
| `algebra-lineal-sheets` | Nuestra librería de matrices (instala sola NumPy y openpyxl) |
| `pandas` | Datos con columnas nombradas — y la réplica de la sección 6 |
| `matplotlib` | Gráficos |
| `ipykernel` | Ejecutar este notebook con el kernel del `.venv` |

> 💡 Si clonaste el repositorio, `pip install -r requirements.txt` instala
> todo esto de una sola vez.

**Paso 4.** En VS Code, arriba a la derecha del notebook haz clic en
**"Select Kernel" / "Seleccionar kernel"** y elige el intérprete del `.venv`.

La siguiente celda verifica que todo quedó instalado:

In [ ]:
# ✅ Verificación del entorno: si esta celda corre sin errores, estás listo
import sys
import numpy as np
import pandas as pd
import matplotlib

from algebra_lineal import *   # nuestra librería de matrices

np.set_printoptions(precision=4, suppress=True)  # impresión legible

print('Python     :', sys.version.split()[0])
print('NumPy      :', np.__version__)
print('pandas     :', pd.__version__)
print('matplotlib :', matplotlib.__version__)
version()

> ❓ **¿Salió `ModuleNotFoundError`?** Casi siempre es una de dos cosas:
> 1. El kernel seleccionado no es el del `.venv` → repite el Paso 4.
> 2. Faltó instalar una librería → activa el entorno y repite el Paso 3.

## 1️⃣ El problema económico

La teoría keynesiana dice que el consumo de los hogares depende del ingreso
disponible; además, esperamos que el precio del bien lo afecte negativamente.
Proponemos el modelo lineal:

$$\text{consumo}_i = \beta_0 + \beta_1\,\text{ingreso}_i + \beta_2\,\text{precio}_i + u_i$$

donde $u_i$ es un término de error. Con una muestra de $n$ observaciones,
nuestro trabajo es **estimar** los coeficientes $\beta_0, \beta_1, \beta_2$:

- $\beta_1$ es la **propensión marginal a consumir** (PMC): ¿cuánto sube el
  consumo si el ingreso sube en 1 unidad, manteniendo el precio constante?
- $\beta_2$ mide el efecto del precio sobre el consumo (esperamos signo
  negativo).

## 2️⃣ Cargar y explorar los datos

`cargar_datos()` lee un `.csv` o `.xlsx` y detecta solo el separador (`,` o
`;`), los decimales con coma y si la primera fila trae los nombres de las
variables. Devuelve un **DataFrame de pandas**: una tabla con columnas
nombradas.

In [ ]:
# Si el CSV no está junto al notebook, lo descargamos del repositorio del curso
import os
if not os.path.exists('datos_consumo.csv'):
    from urllib.request import urlretrieve
    urlretrieve('https://raw.githubusercontent.com/franperezec/'
                'algebra-lineal-sheets/main/ejemplos/datos_consumo.csv',
                'datos_consumo.csv')
    print('📥 datos_consumo.csv descargado')

datos = cargar_datos('datos_consumo.csv')
datos

In [ ]:
datos.describe().round(2)   # estadísticas descriptivas de cada variable

Antes de estimar, **siempre** hay que mirar los datos. Grafiquemos el consumo
contra el ingreso:

In [ ]:
import matplotlib.pyplot as plt

# Estilo sobrio para todos los gráficos de la clase
plt.rcParams.update({
    'figure.figsize': (7, 4.5),
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.grid': True,
    'axes.axisbelow': True,
    'grid.color': '#e1e0d9',
    'grid.linewidth': 0.8,
    'axes.edgecolor': '#c3c2b7',
    'axes.labelcolor': '#52514e',
    'xtick.color': '#898781',
    'ytick.color': '#898781',
})
AZUL, NARANJA = '#2a78d6', '#eb6834'

fig, ax = plt.subplots()
ax.scatter(datos['ingreso'], datos['consumo'], s=60, color=AZUL)
ax.set_xlabel('Ingreso')
ax.set_ylabel('Consumo')
ax.set_title('Consumo vs. ingreso (datos observados)', loc='left')
plt.show()

La relación es claramente **lineal y positiva**: a mayor ingreso, mayor
consumo. Pero el gráfico no lo dice todo — el precio también varía entre
observaciones. Para separar el efecto de cada variable necesitamos la
**regresión múltiple**.

## 3️⃣ Teoría: MCO en forma matricial

Apilamos las $n$ observaciones en matrices:

$$\underbrace{\begin{bmatrix} \text{consumo}_1 \\ \vdots \\ \text{consumo}_n \end{bmatrix}}_{y \;(n\times 1)} = \underbrace{\begin{bmatrix} 1 & \text{ingreso}_1 & \text{precio}_1 \\ \vdots & \vdots & \vdots \\ 1 & \text{ingreso}_n & \text{precio}_n \end{bmatrix}}_{X \;(n\times k)} \underbrace{\begin{bmatrix} \beta_0 \\ \beta_1 \\ \beta_2 \end{bmatrix}}_{\beta \;(k\times 1)} + u$$

es decir, $y = X\beta + u$. La columna de unos en $X$ genera el **intercepto**
$\beta_0$.

El estimador MCO minimiza la suma de los residuos al cuadrado, y su fórmula es
la solución de las **ecuaciones normales**:

$$(X'X)\,\hat\beta = X'y \quad\Longrightarrow\quad \hat\beta = (X'X)^{-1}X'y$$

Todo lo que necesitamos son operaciones que ya conocemos: **transpuesta,
multiplicación de matrices e inversa** (o mejor aún: resolver el sistema).

## 4️⃣ Estimación con nuestra librería

`matriz_diseno()` construye $X$ y $y$ a partir del DataFrame: elige la columna
dependiente, agrega la columna de unos al inicio y devuelve también los
nombres de los coeficientes en orden.

In [ ]:
X, y, nombres = matriz_diseno(datos, y='consumo', x=['ingreso', 'precio'])

print('X es de', X.shape, '→ (n observaciones × k coeficientes)')
print('y es de', y.shape, '→ vector columna')
print('Coeficientes a estimar:', nombres)
print()
print('Primeras filas de X:')
print(X[:4])

Ahora estimamos $\hat\beta$ **paso a paso** con las ecuaciones normales:

In [ ]:
XtX = X.T @ X     # matriz k×k
Xty = X.T @ y     # vector k×1

print("X'X =")
print(XtX)
print()
print("X'y =")
print(Xty)

# Resolver el sistema (X'X) beta = X'y
beta = np.linalg.solve(XtX, Xty)

print()
print('Coeficientes estimados:')
for nombre, valor in zip(nombres, beta.flatten()):
    print(f'   {nombre:>8}: {valor:8.3f}')

> 🧮 La fórmula del pizarrón es $\hat\beta = (X'X)^{-1}X'y$. En el computador
> es preferible `np.linalg.solve(XtX, Xty)`: resuelve el sistema **sin
> calcular la inversa explícita**, que es más costosa y menos precisa
> numéricamente. Verifiquemos que dan lo mismo:

In [ ]:
beta_con_inversa = np.linalg.inv(XtX) @ Xty
print('β̂ con la inversa explícita:', beta_con_inversa.flatten())
print('¿Iguales?', np.allclose(beta, beta_con_inversa))

**Interpretación económica:**

- `const` ≈ **consumo autónomo**: lo que se consumiría con ingreso cero.
- `ingreso` es la **propensión marginal a consumir**: por cada unidad
  adicional de ingreso, el consumo sube ≈ 0.8 unidades (precio constante).
- `precio` tiene signo **negativo**, como esperaba la teoría.

## 5️⃣ ¿Qué tan bien ajusta el modelo?

Con $\hat\beta$ calculamos los **valores ajustados** $\hat y = X\hat\beta$ y
los **residuos** $\hat u = y - \hat y$. El $R^2$ mide la fracción de la
variación del consumo que el modelo explica:

$$R^2 = 1 - \frac{\hat u'\hat u}{\sum_i (y_i - \bar y)^2}$$

In [ ]:
y_ajustado = X @ beta          # valores ajustados
residuos = y - y_ajustado      # residuos

SRC = (residuos.T @ residuos).item()              # suma de residuos al cuadrado
STC = ((y - y.mean()).T @ (y - y.mean())).item()  # suma total de cuadrados
r2 = 1 - SRC / STC

print(f'R² = {r2:.4f} → el modelo explica el {r2:.2%} de la variación del consumo')

In [ ]:
fig, ax = plt.subplots()
ax.scatter(datos['ingreso'], y.flatten(), s=60, color=AZUL,
           label='Consumo observado')
ax.scatter(datos['ingreso'], y_ajustado.flatten(), s=70, color=NARANJA,
           marker='x', linewidths=2, label='Consumo ajustado (ŷ = Xβ̂)')
ax.set_xlabel('Ingreso')
ax.set_ylabel('Consumo')
ax.set_title('Observado vs. ajustado', loc='left')
ax.legend(frameon=False)
plt.show()

## 6️⃣ Réplica "desde cero" con pandas + NumPy

Nada de lo anterior es magia: `cargar_datos()` y `matriz_diseno()` solo nos
ahorraron pasos. Hagamos **exactamente lo mismo** usando pandas y NumPy
directamente — así se ve lo que la librería hace por dentro.

In [ ]:
# 1. Leer el CSV con pandas
datos_pd = pd.read_csv('datos_consumo.csv')

# 2. Construir a mano la matriz de diseño: columna de unos + regresores
n = len(datos_pd)
X_pd = np.column_stack([
    np.ones(n),                              # columna de la constante
    datos_pd[['ingreso', 'precio']].to_numpy(),
])
y_pd = datos_pd['consumo'].to_numpy().reshape(-1, 1)   # vector columna n×1

# 3. Ecuaciones normales, igual que antes
beta_pd = np.linalg.solve(X_pd.T @ X_pd, X_pd.T @ y_pd)

print('β̂ con pandas + NumPy :', beta_pd.flatten())
print('β̂ con algebra_lineal :', beta.flatten())
print('¿Idénticos?', np.allclose(beta, beta_pd))

NumPy también trae un estimador de mínimos cuadrados listo para usar,
`np.linalg.lstsq`, que resuelve el mismo problema con un método numérico más
robusto:

In [ ]:
beta_lstsq, *_ = np.linalg.lstsq(X_pd, y_pd, rcond=None)
print('β̂ con np.linalg.lstsq:', beta_lstsq.flatten())

> 🧪 **Opcional — la herramienta profesional.** En econometría aplicada usarás
> librerías como `statsmodels`, que además calculan errores estándar, pruebas
> $t$, intervalos de confianza, etc. Si quieres probarla, instala en tu
> `.venv`: `pip install statsmodels` y ejecuta la celda siguiente. Los
> coeficientes deben coincidir exactamente con los nuestros.

In [ ]:
try:
    import statsmodels.api as sm
    resultado = sm.OLS(y_pd, X_pd).fit()
    print(resultado.summary(xname=nombres))
except ImportError:
    print('statsmodels no está instalado (celda opcional): pip install statsmodels')

## 7️⃣ Exportar los resultados a Excel

Como en clase, podemos mandar cualquier matriz a un Excel con `exportar()` —
el archivo se crea si no existe. Y con `guardar_datos()` guardamos una tabla
ordenada (DataFrame) en `.csv` o `.xlsx`:

In [ ]:
# La matriz beta → pestaña 'beta' de un Excel nuevo
exportar('beta', sheet_name='mi_regresion.xlsx')

# Y una tabla ordenada con nombres → CSV
tabla = pd.DataFrame({'coeficiente': nombres,
                      'estimacion': beta.flatten().round(4)})
guardar_datos(tabla, 'resultados_consumo.csv')
tabla

> 📊 Si clonaste el repositorio, compara tu `mi_regresion.xlsx` con
> [`resultados_regresion.xlsx`](resultados_regresion.xlsx) (pestaña `beta`):
> deben coincidir.

## 8️⃣ La "verdad" detrás de los datos

Confesión: los datos de esta clase fueron **generados** con la fórmula

$$\text{consumo} = 10 + 0.8\,\text{ingreso} - 1.5\,\text{precio} + \text{ruido}$$

Compara con tu $\hat\beta$: MCO recuperó los parámetros verdaderos con notable
precisión. Con datos reales nunca conocemos la "verdad" — por eso en
econometría importan los supuestos del modelo y las medidas de precisión de
los estimadores.

## ✏️ Ejercicios

1. **Variable omitida.** Estima el modelo solo con el ingreso:
   `matriz_diseno(datos, y='consumo', x=['ingreso'])`. ¿Cambia la PMC
   estimada? ¿Por qué? ¿Y el $R^2$?
2. **Sin constante.** Repite la estimación con `constante=False`. ¿Qué pasa
   con los coeficientes? ¿Tiene sentido económico un consumo autónomo igual a
   cero?
3. **Predicción.** Con tu $\hat\beta$: ¿cuánto consumo predice el modelo para
   `ingreso = 220` y `precio = 6`? *(Pista: arma el vector fila
   $x_0 = [1, 220, 6]$ y calcula $x_0\hat\beta$.)*
4. **Tus propios datos.** Consigue un CSV con una variable dependiente y dos
   regresores (o crea uno en Excel), cárgalo con `cargar_datos()` y estima tu
   propia regresión.